# Task 7: Multi-Agent AI System for Travel Planning (Kaggle, no API key)
Five specialised agents collaborate step by step; each agent's output becomes input for the next:

**Destination** → **Budget** → **Itinerary** → **Recommendation** → **Evaluator** (can trigger a revision)

## Before you run (Kaggle settings)
This notebook uses a **free open-source model that runs inside Kaggle**, so **no API key is needed**.
In the right sidebar → **Session options**:
1. **Accelerator** → *GPU T4 x2* (or *GPU P100*), because on CPU it will be very slow.
2. **Internet** → **On** (needed once to download the model; Kaggle may ask you to verify your phone number).

## 1. Install

In [3]:
!pip install -q langchain-core accelerate

## 2. Load the free LLM

In [4]:
# Free local LLM - NO API key. Runs on the Kaggle GPU.
import torch, transformers
from transformers import pipeline, AutoTokenizer
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import RunnableLambda

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"     # smaller/faster option: "Qwen/Qwen2.5-1.5B-Instruct"

transformers.logging.set_verbosity_error()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
pipe = pipeline("text-generation", model=MODEL_ID, tokenizer=tokenizer,
                torch_dtype=torch.float16, device_map="auto")
print("Model loaded on:", pipe.model.device)

_ROLES = {"system": "system", "human": "user", "ai": "assistant"}

def get_llm(temperature=0.3, max_new_tokens=700):
    """Returns a LangChain runnable: accepts a prompt / list of messages, returns an AIMessage."""
    def _run(x):
        if isinstance(x, str):
            msgs = [HumanMessage(content=x)]
        elif hasattr(x, "to_messages"):
            msgs = x.to_messages()
        else:
            msgs = x
        chat = [{"role": _ROLES.get(m.type, "user"), "content": m.content} for m in msgs]
        prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        kwargs = dict(max_new_tokens=max_new_tokens, return_full_text=False)
        if temperature > 0:
            kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
        else:
            kwargs.update(do_sample=False)
        out = pipe(prompt, **kwargs)[0]["generated_text"]
        return AIMessage(content=out.strip())
    return RunnableLambda(_run)

def text_of(content):
    return content if isinstance(content, str) else "".join(b.get("text", "") for b in content if isinstance(b, dict))

print(get_llm(0).invoke("Say hello in one short sentence.").content)   # quick test

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded on: cuda:0
Hello!


## 3. Define the Agent class and the five agents

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage

class Agent:
    """A specialised worker: a name, a role (system prompt) and a run() method."""
    def __init__(self, name, role, temperature=0.4):
        self.name, self.role = name, role
        self.llm = get_llm(temperature, max_new_tokens=900)

    def run(self, task):
        print(f"  -> {self.name} is working...")
        reply = self.llm.invoke([SystemMessage(content=self.role), HumanMessage(content=task)])
        return text_of(reply.content)

destination_agent = Agent("Destination Agent",
    "You are a destination expert. Analyse the destination and suggest 6-8 places and activities that match the "
    "traveller's interests, budget level and trip length. For each give a one-line reason, the area/neighbourhood, "
    "and an approximate time needed. Use concise markdown bullets.")

budget_agent = Agent("Budget Agent",
    "You are a travel budget analyst. Using the suggested places, estimate costs for accommodation, food, local "
    "transportation and activities. Give per-day and total figures in the traveller's currency in a markdown table, "
    "state assumptions, and say clearly whether the total fits the budget. If it doesn't, suggest where to cut.",
    temperature=0.2)

itinerary_agent = Agent("Itinerary Agent",
    "You are an itinerary planner. Build a realistic day-by-day schedule (Morning / Afternoon / Evening) using ONLY "
    "the suggested places and respecting the budget analysis. Group nearby places on the same day, keep travel time "
    "sensible, and include meal ideas. Use '### Day N' headings.")

recommendation_agent = Agent("Recommendation Agent",
    "You are a senior travel advisor. Review the places, budget and itinerary. Produce final recommendations: best "
    "time/season tips, money-saving tips, local etiquette, safety, what to book in advance, and a short packing list. "
    "Fix any issues you notice in earlier outputs.")

evaluator_agent = Agent("Evaluator Agent",
    "You are a strict QA reviewer. Check the plan for consistency with the user's budget, interests and number of "
    "days. Reply in this exact format:\n"
    "VERDICT: PASS or REVISE\nBUDGET: ok/issue - one line\nINTERESTS: ok/issue - one line\n"
    "TIME: ok/issue - one line\nFIXES: bullet list of required changes (or 'none').", temperature=0)

## 4. Orchestrate the collaboration

In [6]:
def plan_trip(destination, budget, days, interests):
    ctx = f"Destination: {destination}\nTotal budget: {budget}\nTrip length: {days} days\nInterests: {interests}"
    out = {}
    print("Running agents:")

    out["places"] = destination_agent.run(f"Traveller request:\n{ctx}\n\nSuggest places and activities.")
    out["budget"] = budget_agent.run(
        f"Traveller request:\n{ctx}\n\nSuggested places:\n{out['places']}\n\nEstimate the budget.")
    out["itinerary"] = itinerary_agent.run(
        f"Traveller request:\n{ctx}\n\nPlaces:\n{out['places']}\n\nBudget:\n{out['budget']}\n\n"
        f"Create the {days}-day itinerary.")
    out["recommendations"] = recommendation_agent.run(
        f"Traveller request:\n{ctx}\n\nPlaces:\n{out['places']}\n\nBudget:\n{out['budget']}\n\n"
        f"Itinerary:\n{out['itinerary']}\n\nWrite the final recommendations.")
    out["evaluation"] = evaluator_agent.run(
        f"Traveller request:\n{ctx}\n\nPlaces:\n{out['places']}\n\nBudget:\n{out['budget']}\n\n"
        f"Itinerary:\n{out['itinerary']}\n\nRecommendations:\n{out['recommendations']}")

    # Feedback loop: evaluator -> recommendation agent revises once
    if "VERDICT: REVISE" in out["evaluation"].upper():
        print("  (Evaluator requested a revision)")
        out["recommendations"] = recommendation_agent.run(
            f"Traveller request:\n{ctx}\n\nItinerary:\n{out['itinerary']}\n\nBudget:\n{out['budget']}\n\n"
            f"Your previous recommendations:\n{out['recommendations']}\n\nEvaluator feedback:\n{out['evaluation']}\n\n"
            "Rewrite the recommendations so they resolve every issue.")
    return out

## 5. Run it
Change the values below to plan your own trip. This takes a few minutes because 5 agents run one after another.

In [7]:
DESTINATION = "Istanbul, Turkey"
BUDGET = "1500 USD"
DAYS = 4
INTERESTS = "history, local food, photography"

result = plan_trip(DESTINATION, BUDGET, DAYS, INTERESTS)

Running agents:
  -> Destination Agent is working...
  -> Budget Agent is working...
  -> Itinerary Agent is working...
  -> Recommendation Agent is working...
  -> Evaluator Agent is working...


## 6. Final travel plan

In [8]:
from IPython.display import Markdown, display

final_plan = f"""# Travel Plan: {DESTINATION}
*{DAYS} days · budget {BUDGET} · interests: {INTERESTS}*

## 1. Recommended Places & Activities
{result['places']}

## 2. Estimated Budget
{result['budget']}

## 3. Day-by-Day Itinerary
{result['itinerary']}

## 4. Final Recommendations
{result['recommendations']}

## Evaluator Check
{result['evaluation']}
"""
display(Markdown(final_plan))
open("travel_plan.md", "w", encoding="utf-8").write(final_plan)

# Travel Plan: Istanbul, Turkey
*4 days · budget 1500 USD · interests: history, local food, photography*

## 1. Recommended Places & Activities
- **Topkapi Palace & Hagia Sophia** (Istanbul Old City): Explore the rich history and stunning architecture, 2 hours.
- **Grand Bazaar**: Discover local Turkish goods and spices, 3 hours.
- **Galata Tower & Ortaköy Mosque**: Capture picturesque views of the city, 1 hour.
- **Çinili Köfte (Pillow Meatballs) & Balık Ekmek (Fish Sandwich)**: Tasteful local cuisine, 1 hour.
- **Bosphorus Cruise**: Enjoy panoramic views of the city and the Bosphorus Strait, 2 hours.
- **Blue Mosque & Basilica Cistern**: Witness more architectural marvels, 2 hours.
- **Tophane & Fener Neighborhoods**: Experience a vibrant street life and hidden gems, 3 hours.

## 2. Estimated Budget
### Travel Budget Analysis for Istanbul, Turkey

#### Assumptions:
- **Accommodation:** A mid-range hotel in the city center is assumed.
- **Transportation:** Public transport (metro, buses) and walking are used for most activities.
- **Food:** Local cuisine is assumed as the main meal option with some snacks.
- **Activities:** All activities are included in the suggested places.
- **Currency:** USD
- **Budget Duration:** 4 days

#### Daily Breakdown:

| Activity/Place        | Cost Estimate (USD) | Total (USD) |
|-----------------------|---------------------|-------------|
| **Day 1**             |                    |             |
| Topkapi Palace & Hagia Sophia | $25 | $25 |
| Grand Bazaar           | $15 | $40 |
| **Day 2**             |                    |             |
| Galata Tower & Ortaköy Mosque | $10 | $50 |
| Çinili Köfte & Balık Ekmek | $10 | $60 |
| **Day 3**             |                    |             |
| Blue Mosque & Basilica Cistern | $20 | $80 |
| Tophane & Fener Neighborhoods | $20 | $100 |
| **Day 4**             |                    |             |
| Bosphorus Cruise       | $25 | $125 |
| **Total**             |                    | $420 |

#### Total Estimated Costs:
- **Accommodation:** Not explicitly mentioned, but assuming a mid-range hotel in the city center, let's assume an average nightly cost of $75 for 4 nights = $300.
- **Total Estimated Budget:** $420 + $300 = $720

#### Comparison with Traveller's Budget:
- **Traveller's Budget:** $1500
- **Total Estimated Budget:** $720

The total estimated budget of $720 falls well within the $1500 budget. Therefore, the trip fits comfortably within the budget.

### Suggested Adjustments:
Since the total fits comfortably within the budget, no adjustments are necessary. However, if you wish to explore additional activities or stay longer, you can consider reducing the number of meals or activities that are not essential to your interests.

## 3. Day-by-Day Itinerary
### Day 1: Exploring Istanbul's Historical Riches

**Morning:**
- **09:00 AM - Departure from Hotel**: Head to Topkapi Palace & Hagia Sophia, starting early to avoid crowds.
- **11:00 AM - Arrival at Topkapi Palace & Hagia Sophia**: Spend 2 hours exploring these historical sites, including the Topkapi Palace and the Hagia Sophia.
- **13:00 PM - Lunch**: Enjoy a traditional Turkish lunch at a local restaurant near the Hagia Sophia. Try **Çinili Köfte (Pillow Meatballs)** and **Balık Ekmek (Fish Sandwich)**.
- **15:00 PM - Transportation**: Take a taxi or metro to the Grand Bazaar, located just a short walk away.
- **16:00 PM - Grand Bazaar**: Spend 3 hours wandering through the Grand Bazaar, purchasing local goods, spices, and souvenirs.
- **19:00 PM - Farewell to the Bazaar**: Return to the hotel for dinner.
- **Evening:**
- **20:00 PM - Dinner**: Enjoy a traditional Turkish dinner at a local restaurant. Consider **Kebab** or **Pide** (Turkish flatbread).
- **22:00 PM - Nightlife**: Explore the nightlife in the area around the hotel.

### Day 2: Capturing Istanbul’s Beauty

**Morning:**
- **09:00 AM - Departure from Hotel**: Start your day early to visit Galata Tower & Ortaköy Mosque.
- **10:00 AM - Galata Tower & Ortaköy Mosque**: Spend 1 hour at Galata Tower and another 1 hour at Ortaköy Mosque.
- **12:00 PM - Lunch**: Have a light lunch at a café near the Galata Tower or Ortaköy Mosque.
- **13:00 PM - Transportation**: Walk to the Blue Mosque & Basilica Cistern.
- **14:00 PM - Blue Mosque & Basilica Cistern**: Spend 2 hours exploring these architectural wonders.
- **16:00 PM - Transportation**: Take a taxi or metro to the Tophane & Fener Neighborhoods.
- **17:00 PM - Tophane & Fener Neighborhoods**: Spend 3 hours exploring this vibrant neighborhood, known for its street art and cultural diversity.
- **19:00 PM - Farewell to Tophane & Fener**: Return to the hotel for dinner.
- **Evening:**
- **20:00 PM - Dinner**: Enjoy a traditional Turkish dinner at a local restaurant. Consider **Kebab** or **Pide** (Turkish flatbread).
- **22:00 PM - Nightlife**: Explore the nightlife in the area around the hotel.

### Day 3: Bosphorus Cruise and More Exploration

**Morning:**
- **09:00 AM - Departure from Hotel**: Head to the Bosphorus Cruise.
- **10:00 AM - Bosphorus Cruise**: Enjoy a 2-hour cruise along the Bosphorus Strait, taking in panoramic views of the city.
- **12:00 PM - Lunch**: Have a light lunch at a café near the cruise port.
- **13:00 PM - Transportation**: Walk to the Grand Bazaar.
- **14:00 PM - Grand Bazaar**: Spend 3 hours wandering through the Grand Bazaar, purchasing local goods, spices, and souvenirs.
- **16:00 PM - Farewell to the Bazaar**: Return to the hotel for dinner.
- **Evening:**
- **20:00 PM - Dinner**: Enjoy a traditional Turkish dinner at a local restaurant. Consider **Kebab** or **Pide** (Turkish flatbread).
- **22:00 PM - Nightlife**: Explore the nightlife in the area around the hotel.

### Day 4: Relaxation and Reflection

**Morning:**
- **09:00 AM - Departure from Hotel**: Start your day early to visit Topkapi Palace & Hagia Sophia.
- **10:0

## 4. Final Recommendations
Based on your detailed request, here are the final recommendations for your 4-day trip to Istanbul, Turkey:

### Best Time/Season Tips:
- **Best Time to Visit:** April to June or September to October offer pleasant weather and fewer crowds compared to peak summer months.
- **Avoid:** July and August are extremely hot and crowded, so consider visiting in the shoulder seasons for a more comfortable experience.

### Money-Saving Tips:
- **Eat Like a Local:** Opt for street food or local eateries rather than high-end restaurants. Street food stalls are often cheaper and more authentic.
- **Public Transport:** Use public transportation like the metro and buses for getting around the city. Walking is also very efficient.
- **Local Souvenirs:** Purchase souvenirs from local markets instead of tourist shops. Markets like the Grand Bazaar offer a wide variety of items at reasonable prices.
- **Skip Unnecessary Activities:** If you're not interested in certain activities, such as the Blue Mosque or Basilica Cistern, you can save money by skipping them.

### Local Etiquette:
- **Respect Religious Sites:** Dress modestly when visiting religious sites. Women should cover their shoulders and knees.
- **Be Respectful:** Be mindful of locals' customs and traditions. For example, tipping is not customary in Turkey, but leaving a small gratuity can be appreciated.
- **Use Hand Sanitizer:** Keep hand sanitizer handy, especially during the heat and humidity.

### Safety:
- **Stay Alert:** Keep an eye on your belongings, especially in crowded areas. Pickpocketing can occur, particularly in tourist-heavy areas.
- **Trust Your Instincts:** If something feels off, trust your instincts and avoid it.

### What to Book in Advance:
- **Flights:** Book flights well in advance, especially during peak seasons.
- **Accommodation:** Book your hotel or Airbnb in advance, especially if you prefer a specific location or type of accommodation.
- **Activities:** Book your Bosphorus cruise and any other activities that require reservations in advance, especially during peak times.

### Packing List:
- **Clothing:** Comfortable walking shoes, lightweight clothing suitable for warm weather, a raincoat or umbrella, and a hat to protect against sun.
- **Toiletries:** Basic toiletries, sunscreen, and a reusable water bottle.
- **Camera:** Bring a camera or smartphone with good photo capabilities.
- **Travel Documents:** Passport, visa, travel insurance, and important documents.
- **Essentials:** Snacks, medications, and any personal items you might need.

By following these recommendations, you can ensure a safe, enjoyable, and budget-friendly trip to Istanbul. Enjoy your exploration of this fascinating city!

## Evaluator Check
VERDICT: PASS
BUDGET: ok - The total estimated costs ($720) fall well within the $1500 budget.
INTERESTS: ok - The itinerary covers history, local food, and photography, aligning with the traveler's interests.
TIME: ok - The 4-day itinerary is feasible given the planned activities and daily commitments.


8980